In [51]:
import polars
import polars as pl
import numpy as np
DATA_BASE_PATH = "data/"

In [52]:
data = polars.read_csv(DATA_BASE_PATH + "node_information.csv")

In [53]:
# Read txt file containing the edges 
edges_df = polars.read_csv(DATA_BASE_PATH + "train.txt", separator=" ", has_header=False, new_columns=["a", "b", "label"])
edges_df.shape

(10496, 3)

### Feature source

In [54]:
# Create node_feature_mapping from node_id to feature vector
node_feature_mapping = {}
for row in data.iter_rows():
    node_id = row[0]
    features = np.array(row[1:])
    node_feature_mapping[node_id] = features

Standard interface to create dataset

In [55]:
def build_dataset(edges, labels, feature_builder):
    X = []
    Y = []
    unseen_nodes_count = 0
    for i in range(edges.shape[0]):
        a, b = edges[i]
        edge_features = feature_builder(a, b)
        if edge_features is not None:
            X.append(edge_features)
            Y.append(labels[i])
        else:
            unseen_nodes_count += 1
    print(f"Rows with >=1 unseen node: {unseen_nodes_count} / {edges.shape[0]}")
    return np.array(X), np.array(Y)

### Feature engineering

In [56]:
def raw_feature_builder(node_a, node_b):
    features_a = node_feature_mapping.get(node_a)
    features_b = node_feature_mapping.get(node_b)
    if features_a is None or features_b is None:
        return None 
    return np.concatenate([features_a, features_b])

In [57]:
data = edges_df.to_numpy()
edges = data[:, :2]  # columns: a, b
labels = data[:, 2]  # column: label

X, Y = build_dataset(edges, labels, raw_feature_builder)

# X, Y = raw_diff_dataset()
print(X.shape, Y.shape)
# Split test and train
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

Rows with >=1 unseen node: 7 / 10496
(10489, 1864) (10489,)


In [58]:
from sklearn.metrics import classification_report

def evaluate_model(model, x, y,) :
    y_predicted = model.predict(x)
    report = classification_report(y, y_predicted)
    print(report)
    

### SVM baseline

In [59]:
from sklearn.svm import SVC

clf = SVC()
clf.fit(X_train, Y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [60]:
evaluate_model(clf, X_test, Y_test)

              precision    recall  f1-score   support

           0       0.60      0.73      0.66      1554
           1       0.67      0.52      0.59      1593

    accuracy                           0.63      3147
   macro avg       0.63      0.63      0.62      3147
weighted avg       0.63      0.63      0.62      3147



### XGboost baseline

In [61]:
from xgboost import XGBClassifier

clf = XGBClassifier()
clf.fit(X_train, Y_train)



,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [62]:
evaluate_model(clf, X_test, Y_test)

              precision    recall  f1-score   support

           0       0.58      0.77      0.66      1554
           1       0.67      0.46      0.54      1593

    accuracy                           0.61      3147
   macro avg       0.63      0.61      0.60      3147
weighted avg       0.63      0.61      0.60      3147



### Logisitic regression

In [63]:
## Logistic Regression
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, Y_train)
evaluate_model(clf, X_test, Y_test)

              precision    recall  f1-score   support

           0       0.60      0.64      0.62      1554
           1       0.62      0.58      0.60      1593

    accuracy                           0.61      3147
   macro avg       0.61      0.61      0.61      3147
weighted avg       0.61      0.61      0.61      3147



In [64]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from karateclub import DeepWalk
import networkx as nx

edges_np = edges_df.to_numpy()
edges_train_np, edges_test_np = train_test_split(
    edges_np,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=edges_np[:, 2],
)

edges_train = pl.DataFrame(edges_train_np, schema=["a", "b", "label"]).with_columns(
    pl.col("a").cast(pl.Int64),
    pl.col("b").cast(pl.Int64),
    pl.col("label").cast(pl.Int64),
)
edges_test = pl.DataFrame(edges_test_np, schema=["a", "b", "label"]).with_columns(
    pl.col("a").cast(pl.Int64),
    pl.col("b").cast(pl.Int64),
    pl.col("label").cast(pl.Int64),
)

print("train/test sizes:", edges_train.shape, edges_test.shape)

train_pos = edges_train.filter(pl.col("label") == 1)
unique_nodes = set(train_pos["a"].to_list()) | set(train_pos["b"].to_list())
node_mapping = {node_id: i for i, node_id in enumerate(sorted(unique_nodes))}

G = nx.Graph()
for row in train_pos.iter_rows(named=True):
    u = node_mapping[row["a"]]
    v = node_mapping[row["b"]]
    G.add_edge(u, v)

print(f"Graph nodes (train positives): {G.number_of_nodes()}")
print(f"Graph edges (train positives): {G.number_of_edges()}")

model = DeepWalk()
model.fit(G)
embeddings = model.get_embedding()
emb_dim = embeddings.shape[1]
print("Embeddings shape:", embeddings.shape)

def get_node_embedding(node_id):
    mapped_id = node_mapping.get(node_id)
    if mapped_id is None:
        return np.ones(emb_dim, dtype=np.float32)
    return embeddings[mapped_id]

def embedding_feature_builder(node_a, node_b):
    emb_a = get_node_embedding(node_a)
    emb_b = get_node_embedding(node_b)
    return np.concatenate([emb_a, emb_b])

def hybrid_feature_builder(node_a, node_b):
    raw_a = node_feature_mapping.get(node_a)
    raw_b = node_feature_mapping.get(node_b)
    if raw_a is None or raw_b is None:
        return None 
    emb_a = get_node_embedding(node_a)
    emb_b = get_node_embedding(node_b)
    emb = np.array([np.dot(emb_a, emb_b)])
    return np.concatenate([raw_a, raw_b, emb_a, emb_b])



X_train_emb, Y_train_emb = build_dataset(edges_train.to_numpy()[:, :2], edges_train.to_numpy()[:, 2], hybrid_feature_builder)
X_test_emb, Y_test_emb = build_dataset(edges_test.to_numpy()[:, :2], edges_test.to_numpy()[:, 2], hybrid_feature_builder)
print("X_train/X_test:", X_train_emb.shape, X_test_emb.shape)


ModuleNotFoundError: No module named 'karateclub'

In [ ]:
clf = XGBClassifier()
clf.fit(X_train_emb, Y_train_emb)
evaluate_model(clf, X_test_emb, Y_test_emb)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, Y_train_emb)
evaluate_model(clf, X_test_emb, Y_test_emb)

              precision    recall  f1-score   support

           0       0.56      0.93      0.70       524
           1       0.79      0.27      0.40       525

    accuracy                           0.60      1049
   macro avg       0.67      0.60      0.55      1049
weighted avg       0.67      0.60      0.55      1049

              precision    recall  f1-score   support

           0       0.58      0.66      0.62       524
           1       0.60      0.52      0.56       525

    accuracy                           0.59      1049
   macro avg       0.59      0.59      0.59      1049
weighted avg       0.59      0.59      0.59      1049



# GNN 

## GCN 

utilisation de Pytorch 
1. create the dataset using PyTorch 
    - build the features, features[i] = feature_of_node_i
    - build the edge index, (edge_index[0][i],edge_index[1][j]) = (idx_of_u_edge,idx_of_v_edge)
2. create the model pipeline 
    - encoder for each node, one embedding per node : GCN 
    - a decoder that takes two nodes embeddings, $(z_i,z_j)$, and outputs a score that is the probability that 
    and edge exist between the two 
3. train with a BCE loss for negative and positive samples 


In [67]:
import torch 
from torch_geometric.data import Data


# create the feature dataset 
# re-index the features 
X_gcn = []
remapping = {}
# create the features 
for i, row in enumerate(data) : 
    node_id = row[0]
    features = np.array(row[1:],dtype=np.float32)
    remapping[node_id] = i 
    X_gcn.append(features)
X_gcn = torch.tensor(np.array(X_gcn), dtype=torch.float) # features 

receiver = []
sender = []
edge_labels = []

for edge, label in zip(edges,labels) : 
    node_i, node_j = edge[0], edge[1]
    if node_i not in remapping or node_j not in remapping:
        continue

    i = remapping[node_i]
    j = remapping[node_j]
    edge_labels.append(label)

    sender.append(i)
    receiver.append(j)


edge_index = torch.tensor([sender,receiver],dtype=torch.long)
edge_label = torch.tensor(edge_labels, dtype=torch.float)

graph_data = Data(
    x=X_gcn,
    edge_index=edge_index,
    edge_label_index=edge_index,
    edge_label=edge_label
)

split according to edges 

In [ ]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True,
)

train_data, val_data, test_data = transform(graph_data)

we can check the number of positive edge vs negative edge, the ratio should be ideally 50/50, for that we can add/remove negative edges if needed 

Create the model 

In [ ]:
import torch.nn.functional as F 
from torch import nn 
from torch_geometric.nn import GCNConv

# create the encoder 
class GCNencoder(nn.Module) : 
    def __init__(self,in_channels,hidden_channels,out_channels) :
        super().__init__
        self.conv1 = GCNConv(in_channels,hidden_channels)
        self.conv2 = GCNConv(hidden_channels,out_channels)
    
    def forward(self,x,edge_index) :
        x = self.conv1(x,edge_index)
        x = F.relu(x)
        x = self.conv2(x,edge_index)
        return x 

# create the decoder
class DotProductDecoder(nn.Module) : 
    def forward(self, z, edge_label_index) : 
        src, dst = edge_label_index
        return (z[src]*z[dst]).sum(dim=1)
    
# variant : MLP decoder, more powerful

class MLPdecoder(nn.Module) :
    def __init__(self,emb_dim,hidden_dim):
        super().__init__
        self.mlp = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, z, edge_label_index):
        src, dst = edge_label_index
        h = torch.cat([z[src], z[dst]], dim=-1)
        return self.mlp(h).squeeze(-1)
 
# create the LinkPredictor : (u,v) -> {0,1} (presence or not)
# assembling the encoder and the decoder 
class LinkPredictor(nn.Module) :
    def __init__(self, in_channels, hidden_channels, out_channels) :
        super().__init__()
        self.encoder = GCNencoder(in_channels,hidden_channels,out_channels)
        self.decoder = DotProductDecoder()

    def forward(self, x, edge_index, edge_label_index):
        z = self.encoder(x, edge_index)
        logits = self.decoder(z, edge_label_index)
        return logits



let's write the training loop

In [ ]:
from torch.optim import AdamW
from sklearn.metrics import roc_auc_score, average_precision_score

# device
device = torch.device("mps")

# instantiate the model
in_channels = train_data.x.size(1)
hidden_channels = 32
out_channels = 16
model = LinkPredictor(in_channels, hidden_channels, out_channels).to(device)

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

optimizer = AdamW(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

def train():
    model.train()
    optimizer.zero_grad()

    logits = model(
        train_data.x,
        train_data.edge_index,
        train_data.edge_label_index
    )

    loss = criterion(logits, train_data.edge_label.float())
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(split_data):
    model.eval()
    logits = model(
        split_data.x,
        split_data.edge_index,
        split_data.edge_label_index
    )
    probs = torch.sigmoid(logits).cpu().numpy()
    y_true = split_data.edge_label.cpu().numpy()
    auc = roc_auc_score(y_true, probs)
    ap = average_precision_score(y_true, probs) # ? 
    return auc, ap

In [ ]:
for epoch in range(1, 101):
    loss = train()
    if epoch % 10 == 0:
        val_auc, val_ap = evaluate(val_data)
        print(f"epoch {epoch:03d}, loss: {loss:.4f}, Val AUC: {val_auc:.4f}, Val AP: {val_ap:.4f}")

test_auc, test_ap = evaluate(test_data)
print(f"Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")